# Semestrální projekt (Část 1): Exploratory Data Analysis & Clustering
**Student:** Diana Kasymbekova (KAS0301)
**Dataset:** Melbourne Housing Snapshot

---

## 1. Úvod a cíle projektu

V této části projektu se zaměříme na analýzu dat (EDA) a shlukování (clustering) nemovitostí v Melbourne.

Naším cílem je prozkoumat data a splnit následující úkoly:

1. **Data Cleaning & Preprocessing:** Vyčištění dat, zpracování chybějících a neobvyklých hodnot.
2. **Feature Engineering:** Vytvoření 2–3 nových proměnných podle vlastností trhu s nemovitostmi.
3. **Deep EDA:** Analýza rozdělení dat, vztahů mezi proměnnými a korelací. Každý graf doplníme vysvětlením.
4. **Cluster Analysis:** Rozdělení nemovitostí do skupin pomocí shlukovacích algoritmů a matematické zdůvodnění počtu skupin.

In [7]:
# Import základních knihoven pro práci s daty
import pandas as pd
import numpy as np

# Import knihoven pro vizualizaci
import matplotlib.pyplot as plt
import seaborn as sns

## 1.1. Načtení a základní kontrola dat (Dataset Validation)
Podle požadavků projektu musí vybraný dataset splňovat přísná kritéria:
* Minimálně 2000 řádků a 10 příznaků.
* Přítomnost spojitých (continuous) a kategorických proměnných.
* Data musí být "surová" (přítomnost chybějících hodnot a nezpracovaných formátů).

Načteme data a programově potvrdíme, že dataset splňuje všechny podmínky.

In [13]:
# Načtení datasetu (soubor musí být ve stejné složce jako notebook)
df = pd.read_csv("melb_data.csv")

# Kontrola velikosti tabulky
print("=== VELIKOST DATASETU ===")
print(f"Počet pozorování (řádků): {df.shape[0]}")
print(f"Počet příznaků (sloupců): {df.shape[1]}\n")

# Zobrazení prvních 3 řádků pro vizuální seznámení se strukturou
display(df.head(3))

=== VELIKOST DATASETU ===
Počet pozorování (řádků): 13580
Počet příznaků (sloupců): 21



,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0


### 1.2. Analýza datových typů a detekce "špinavých" dat
Abychom mohli naplánovat strategie čištění (imputation strategies), musíme zjistit, ve kterých příznacích chybí hodnoty a jaké datové typy (např. data prodeje) jsou uloženy jako běžný text.

In [12]:
# Kontrola datových typů ve sloupcích
print("=== DATOVÉ TYPY A POČET VYPLNĚNÝCH HODNOT ===")
df.info()

print("\n" + "="*50 + "\n")

# Výpočet prázdných hodnot (zobrazíme pouze sloupce, kde reálně chybí data)
print("=== POČET CHYBĚJÍCÍCH HODNOT (NaN) VE SLOUPCÍCH ===")
missing_values = df.isna().sum().sort_values(ascending=False)
missing_only = missing_values[missing_values > 0]

# Vytvoření přehledné tabulky s procentuálním podílem chybějících dat
missing_df = pd.DataFrame({
    'Chybějící hodnoty': missing_only,
    'Procento z celku (%)': (missing_only / len(df) * 100).round(2)
})

display(missing_df)

=== DATOVÉ TYPY A POČET VYPLNĚNÝCH HODNOT ===
<class 'pandas.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  str    
 1   Address        13580 non-null  str    
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  str    
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  str    
 6   SellerG        13580 non-null  str    
 7   Date           13580 non-null  str    
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  str    
 17  Lattitude      

,Chybějící hodnoty,Procento z celku (%)
BuildingArea,6450,47.50
YearBuilt,5375,39.58
CouncilArea,1369,10.08
Car,62,0.46


### 2. Zpracování chybějících hodnot (Data Cleaning & Imputation)

**2.1. Zpracování sloupce `BuildingArea` (Obytná plocha)**
Sloupec `BuildingArea` vykazuje největší ztrátu dat – chybí zde **6 450 hodnot** (téměř 47,5 %). 

Nelze použít jednoduché nahrazení průměrem (default mean imputation). Pokud bychom nahradili téměř polovinu hodnot jedním průměrem, výrazně bychom snížili rozptyl a ztratili přirozené rozdíly mezi daty. Sloupec také nemůžeme odstranit, protože obytná plocha je důležitá pro analýzu trhu s nemovitostmi i pro budoucí clustering.

**Zvolená strategie (Justified Strategy):** 
Využijeme logiku trhu – velikost obytné plochy souvisí s počtem pokojů (`Rooms`) a typem nemovitosti (`Type` – dům, byt, townhouse). Provedeme seskupení dat podle těchto dvou příznaků a chybějící plochu nahradíme **mediánem dané specifické skupiny**. Tím zajistíme, že chybějící plocha 3pokojového bytu bude nahrazena typickou velikostí 3pokojového bytu, nikoliv průměrem celého města.

In [16]:
# 1. Zkontrolujeme původní stav chybějících hodnot
print("Původní počet chybějících hodnot v BuildingArea:", df['BuildingArea'].isna().sum())

# 2. Nahrazení chybějících hodnot mediánem podle typu nemovitosti a počtu pokojů
# Funkce transform('median') vypočítá medián pro každou skupinu a přiřadí ho k jednotlivým řádkům.
df['BuildingArea'] = df['BuildingArea'].fillna(df.groupby(['Type', 'Rooms'])['BuildingArea'].transform('median'))

# 3. Zkontrolujeme výsledek
zbyle_nan = df['BuildingArea'].isna().sum()
print(f"Zbývající počet chybějících hodnot v BuildingArea po skupinové imputaci: {zbyle_nan}")

Původní počet chybějících hodnot v BuildingArea: 2
Zbývající počet chybějících hodnot v BuildingArea po skupinové imputaci: 2


### 3.2. Zpracování sloupce `YearBuilt` (Rok výstavby)

Ve sloupci `YearBuilt` chybělo více než 5 300 hodnot. Nahrazení všech chybějících hodnot jedním průměrem by mohlo zkreslit data.

Domy ve stejné čtvrti (`Suburb`) a stejného typu (`Type`) mohou mít podobný rok výstavby.

**Strategie:** Chybějící hodnoty nahradíme mediánem podle čtvrti a typu nemovitosti. Pokud některé hodnoty stále chybí, použijeme medián podle typu nemovitosti.

### 3.3. Formátování data (`Date`) a finální čištění

Sloupec `Date` obsahuje data ve formě textu (`object`). Převedeme je na typ `datetime`, abychom mohli analyzovat časové trendy.

Nakonec odstraníme zbývající řádky s chybějícími hodnotami, například ve sloupcích `CouncilArea` a `BuildingArea`. Protože jich je už jen málo, jejich odstranění nebude mít velký vliv na celkovou analýzu.

In [17]:
# 1. Imputace YearBuilt pomocí mediánu podle čtvrti a typu
df['YearBuilt'] = df['YearBuilt'].fillna(
    df.groupby(['Suburb', 'Type'])['YearBuilt'].transform('median')
)

# 2. Doplnění zbývajících chybějících hodnot v YearBuilt podle typu nemovitosti
df['YearBuilt'] = df['YearBuilt'].fillna(
    df.groupby('Type')['YearBuilt'].transform('median')
)

# 3. Parsování textového data na datetime formát (v datech je formát DD/MM/YYYY)
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')

# 4. Odstranění všech zbylých řádků s jakoukoliv chybějící hodnotou (např. CouncilArea)
df_clean = df.dropna().copy()

# 5. Kontrola výsledku po čištění
print("=== KONTROLA DAT PO ČIŠTĚNÍ ===")
print(f"Počet chybějících hodnot v celém datasetu: {df_clean.isna().sum().sum()}")
print(f"Konečný počet řádků připravených pro analýzu: {len(df_clean)}")

=== KONTROLA DAT PO ČIŠTĚNÍ ===
Počet chybějících hodnot v celém datasetu: 0
Konečný počet řádků připravených pro analýzu: 12209
